In [ ]:
import sys
sys.path.append("src")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from degradation import degrade_frame
from optical_flow import frames_to_flow_rgb

In [ ]:
CSV_PATH = "data/splits/manifest.csv"
df = pd.read_csv(CSV_PATH)

row = df.iloc[0]
video_path = row["path"]

print(row)
print(video_path)

In [ ]:
def read_sample_frames(video_path, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    indices = np.linspace(0, total - 1, num_frames).astype(int)

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()

        if not ok:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

frames = read_sample_frames(video_path, num_frames=8)

print("Num frames:", len(frames))
print("Frame shape:", frames[0].shape)

In [ ]:
frame = frames[0]
degraded = degrade_frame(frame, level="strong")

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(frame)
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(degraded)
plt.title("Strong Degradation")
plt.axis("off")

plt.show()

In [ ]:
flow_frames = frames_to_flow_rgb(frames)

print("Num flow maps:", len(flow_frames))

plt.figure(figsize=(5, 5))
plt.imshow(flow_frames[0])
plt.title("Optical Flow RGB Map")
plt.axis("off")
plt.show()

In [ ]:
import os

os.makedirs("outputs/figures", exist_ok=True)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(frame)
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(degraded)
plt.title("Degraded RGB")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(flow_frames[0])
plt.title("Optical Flow")
plt.axis("off")

plt.tight_layout()
plt.savefig("outputs/figures/preprocess_demo.png", dpi=300, bbox_inches="tight")
plt.show()